In [1]:
from pathlib import Path
import numpy as np
from Bio import SeqIO
from scipy.io import savemat

# ===== 1. 路径设置 =====
ROOT = Path(r"E:\MYS\GRU4ACE")
POS_FASTA = ROOT / "Datasets" / "HighActivity.fasta"
NEG_FASTA = ROOT / "Datasets" / "LowActivity.fasta"
OUT_DIR = ROOT / "outputs"
OUT_DIR.mkdir(exist_ok=True)

# ===== 2. 读取 FASTA =====
def read_fasta_sequences(fasta_path):
    seqs = []
    for record in SeqIO.parse(str(fasta_path), "fasta"):
        seqs.append(str(record.seq).strip().upper())
    return seqs

pos_seqs = read_fasta_sequences(POS_FASTA)
neg_seqs = read_fasta_sequences(NEG_FASTA)

sequence = np.array(pos_seqs + neg_seqs, dtype=object)
label = np.array([1] * len(pos_seqs) + [0] * len(neg_seqs), dtype=int)

print(f"positive: {len(pos_seqs)}")
print(f"negative: {len(neg_seqs)}")
print(f"total: {len(sequence)}")

# ===== 3. 定义氨基酸字典 =====
# 20种标准氨基酸 + 1个 padding/unknown
AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"
aa_to_idx = {aa: i for i, aa in enumerate(AMINO_ACIDS)}
PAD_IDX = 20   # 第21类，用于 padding 或未知字符

# 先用论文里的最大长度 21 试跑
MAX_LEN = 21

# ===== 4. 把每条序列转成固定长度的整数序列 =====
def exchange_matrix(protein, max_len=MAX_LEN):
    protein = protein.upper()
    idxs = [aa_to_idx.get(aa, PAD_IDX) for aa in protein[:max_len]]
    idxs += [PAD_IDX] * (max_len - len(idxs))
    return np.array(idxs, dtype=np.int32)

output1 = []
for protein in sequence:
    output = exchange_matrix(protein)
    output1.append(output)

matrix = np.vstack(output1)   # shape: (样本数, MAX_LEN)
num1, column = matrix.shape

print("matrix shape:", matrix.shape)

# ===== 5. 做 one-hot，得到 BPF 特征 =====
vec = []

for i in range(num1):
    A = matrix[i, :]
    vector = []

    for j in range(column):
        feature = np.zeros(21, dtype=np.int8)
        feature[int(A[j])] = 1
        vector.extend(feature)

    vec.append(vector)

matrix_two = np.array(vec, dtype=np.int8)

print("BPF feature shape:", matrix_two.shape)

# ===== 6. 保存 =====
savemat(OUT_DIR / "BPF_all.mat", {
    "matrix_two": matrix_two,
    "label": label,
    "sequence": sequence
})

print("saved to:", OUT_DIR / "BPF_all.mat")

positive: 394
negative: 626
total: 1020
matrix shape: (1020, 21)
BPF feature shape: (1020, 441)
saved to: E:\MYS\GRU4ACE\outputs\BPF_all.mat


In [2]:
print("matrix shape:", matrix.shape)
print("BPF feature shape:", matrix_two.shape)

matrix shape: (1020, 21)
BPF feature shape: (1020, 441)


In [3]:
matrix_two.shape

(1020, 441)